# Kalman Historical V2 Analysis\n\n기존 Historical run `20260913_042850`을 재실행하지 않고 분석합니다.\n\n- Qlib experiment recording\n- vectorbt independent validation\n- Equal Weight / Inverse Vol / HRP / Risk Parity / CVaR / CDaR benchmark\n- Walk-forward OOS permutation feature contribution\n- immutable artifact provenance + pinned Git SHA\n\n**Safety:** Research only / Toss execution OFF / Neon write OFF\n

In [ ]:
from google.colab import drive\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nPINNED_SHA = "921c70ed66f8a73ecd655637e39234ea2e7d75e4"\nRUN_TAG = "20260913_042850"\n\ndrive.mount('/content/drive', force_remount=False)\n\nrepo = Path('/content/Codex')\nif repo.exists():\n    shutil.rmtree(repo)\nrepo.mkdir(parents=True)\n\nsubprocess.run(['git', '-C', str(repo), 'init'], check=True)\nsubprocess.run(['git', '-C', str(repo), 'remote', 'add', 'origin', 'https://github.com/kimtk94/Codex.git'], check=True)\nsubprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', PINNED_SHA], check=True)\nsubprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)\n\ngateway = repo / 'kalman-toss-gateway'\nrunner = gateway / 'scripts' / 'colab_historical_v2_analysis.py'\nassert runner.exists(), runner\n\nsubprocess.run(\n    [\n        'python', str(runner),\n        '--drive-root', '/content/drive/MyDrive',\n        '--run-tag', RUN_TAG,\n        '--pinned-code-sha', PINNED_SHA,\n        '--permutation-repeats', '3',\n    ],\n    check=True,\n)\n\nsummary = Path('/content/drive/MyDrive/Market_Model_V2/historical_quant_2017_v1') / RUN_TAG / 'analysis_v2' / 'historical_v2_analysis_summary.json'\nprint('\nSUMMARY:', summary)\nif summary.exists():\nrepo = Path('/content/Codex')
if repo.exists():
    shutil.rmtree(repo)

# Full clone first, then checkout the pinned commit. This avoids Colab/GitHub
# failures when fetching an arbitrary SHA directly from a shallow repository.
subprocess.run(
    ['git', 'clone', '--branch', 'main', 'https://github.com/kimtk94/Codex.git', str(repo)],
    check=True,
)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA], check=True)

# Basic integrity check before creating any analysis environment.
checked = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert checked == PINNED_SHA, (checked, PINNED_SHA)
subprocess.run(
    [
        'python', '-m', 'py_compile',
        str(repo / 'kalman-toss-gateway' / 'scripts' / 'colab_historical_v2_analysis.py'),
        str(repo / 'kalman-toss-gateway' / 'research' / 'quant_stack' / 'historical_v2_analysis.py'),
        str(repo / 'kalman-toss-gateway' / 'research' / 'quant_stack' / 'historical_v2_vectorbt.py'),
        str(repo / 'kalman-toss-gateway' / 'research' / 'quant_stack' / 'historical_v2_riskfolio.py'),
    ],
    check=True,
)

    print(summary.read_text(encoding='utf-8'))\n